OpenAI Agents SDK

Trip Planner — Example Agent Notebook

This notebook demonstrates how to build a simple trip-planning agent using the OpenAI Agents SDK. It walks through environment setup, defining tools, composing an `Agent`, running the agent with `Runner` and `trace`, and persisting the final plan as a Markdown file in an output sandbox.

What this notebook shows:
- Configure and load a `.env` for API keys and secrets.
- Create tool helpers (web search, simple `function_tool` save/read helpers, and an MCP filesystem server) to extend agent capabilities.
- Build a `Trip Planner Agent` that uses those tools to plan a multi-day trip and save the result.
- Run the agent asynchronously and inspect the trace for debugging and auditability.

Tools
- `WebSearchTool`: performs web lookups (configurable `search_context_size`).
- `function_tool` helpers: save and read Markdown files under the `output/` folder.
- MCP filesystem (`MCPServerStdio`): provides a sandboxed filesystem tool for secure read/write operations.

Agent
A single `Trip Planner Agent` is provided as an example. It accepts plain-language directions (e.g., "plan a 5 day trip to Paris") and demonstrates tool use and file persistence. The final trip plan is written to `output/trip_plan.md` by default.

How to run this notebook:
1. Place a `.env` containing required API keys in the repository root or a parent directory so the notebook can discover it.
2. Run the cells in order. The notebook uses async `Runner.run(...)` calls — run in an async-capable Jupyter environment or adapt the code into an async script.
3. Inspect the agent trace at https://platform.openai.com/traces to review tool calls and model outputs.

Notes
- Outputs are written to the `output/` folder. The notebook includes example `function_tool` wrappers that save and read Markdown files.
- MCP-based filesystem usage launches an external Node MCP server; ensure `npx` and network permissions are available if you run that cell.
- Only the OPENAI_API_KEY parameter needs to be set in order to run this notebook.


In [ ]:
import os
from dotenv import load_dotenv, find_dotenv, dotenv_values
from agents import Agent, Runner, Tool, WebSearchTool, trace, function_tool

In [2]:
# Locate .env in this directory or any parent directory
dotenv_path = find_dotenv()
if not dotenv_path:
    raise FileNotFoundError('.env not found in repository or parent directories')

# Load into os.environ (preserves existing variables unless overridden by .env)
load_dotenv(dotenv_path, override=False)
# Also read raw values as a dict (useful to expose into notebook globals)
env = {k: v for k, v in dotenv_values(dotenv_path).items() if v is not None}

# Export into notebook globals for easy access by name
globals().update(env)

print('Loaded .env from', dotenv_path)
print('Loaded keys:', list(env.keys()))

Loaded .env from /media/nathan/linux_ssd/github/agentic_ai_trip_planner/.env
Loaded keys: ['OPENAI_API_KEY', 'GROQ_API_KEY', 'PUSHOVER_USER', 'PUSHOVER_TOKEN', 'SENDGRID_API_KEY', 'GOOGLE_API_KEY', 'SERPER_API_KEY', 'LANGSMITH_TRACING', 'LANGSMITH_ENDPOINT', 'LANGSMITH_API_KEY', 'LANGSMITH_PROJECT', 'POLYGON_API_KEY', 'POLYGON_PLAN', 'BRAVE_API_KEY']


### A simple agent to plan a basic trip.

A minimal, runnable trip-planner example (see the next code cell). 

- Sets basic input variables: `duration`, `destination`, and a list of `activities` used as planning prompts.
- Instantiates `WebSearchTool(search_context_size="low")` so the agent can perform brief web lookups.
- Constructs a `Trip Planner Agent` using the web-search tool as its only tool in this example.
- Builds a `directions` string that asks the agent to plan a trip using the provided inputs.
- Runs the agent with `await Runner.run(trip_planner_agent, directions)` inside a `trace(...)` context and prints `result.final_output`.
- After running the cell, go to https://platform.openai.com/traces to view the traces for insights into the actions that the agent performed.


In [6]:
# Definding a simple trip planner agent that uses a web search tool to find relevant information about the trip.
duration = "5 days"
destination = "Paris"
activities = ["sightseeing", "dining", "museums"]
directions = f"plan a {duration} trip to {destination} including these activities: {', '.join(activities)}. Use the web search tool to find relevant information. Produce a detailed itinerary."

# Create the web search tool with low context size to limit costs
web_search_tool = WebSearchTool(search_context_size="low")

# Define the trip planner agent
trip_planner_agent = Agent(
    name="Trip Planner Agent",
    instructions="An agent that helps users plan trips by searching for destinations, accommodations, and activities.",
    tools=[
        web_search_tool
    ]
)

# Run the trip planner agent
with trace("Trip Planner Agent - Simple one tool usage"):
    result = await Runner.run(trip_planner_agent, directions)
    print(result.final_output)


Here is a detailed, thoughtfully paced 5‑day Paris itinerary that balances sightseeing, museum visits, and memorable dining experiences. All activities reflect up‑to‑date information for late December 2025.

---

Day 1 – Monuments & History  
Morning  
• Begin at the Louvre Museum to see highlights like the Mona Lisa and Winged Victory, spanning ancient to Renaissance art ([haussmann.galerieslafayette.com](https://haussmann.galerieslafayette.com/en/paris-itinerary-5-days/?utm_source=openai)).  

Afternoon  
• Walk to the Centre Pompidou. **Note**: the Centre Pompidou is *completely closed for renovation until 2030* ([en.wikipedia.org](https://en.wikipedia.org/wiki/Centre_Pompidou?utm_source=openai)). Instead, visit the nearby **Fondation Cartier pour l’Art Contemporain**, now reopened in a historic Palais‑Royal building with dynamic contemporary exhibits ([vogue.com](https://www.vogue.com/article/the-fondation-cartier-is-fronting-a-new-era-for-contemporary-art-in-paris?utm_source=opena